In [1]:
# Libraries
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker

In [23]:
# ================================
# File list
# ================================
files = [
    "/home/aaccardo/these_alex/Papers/Rising_stars/Data/Turbulence/APERO_THA_KvEps_031.nc",
    "/home/aaccardo/these_alex/Papers/Rising_stars/Data/Turbulence/APERO_THA_KvEps_073.nc",
    "/home/aaccardo/these_alex/Papers/Rising_stars/Data/Turbulence/APERO_THA_KvEps_090.nc",
]

# ================================
# Constants
# ================================
nu = 1e-6  # kinematic viscosity [m2/s]
sec_per_day = 86400.0

label_size = 80
tick_size = 100
pad = 50
line_w = 8
grid_line = 5
plot_line_w = 10
depth_max = 1500

# ================================
# Titles for each row
# ================================
row_titles = [
    "PSS1",
    "PSS3",
    "PSS4",
]

# ================================
# Prepare figure
# ================================
fig, axes = plt.subplots(
    nrows=3, ncols=4, figsize=(100, 100), sharey=True, sharex = "col"
)

# --- Set y-ticks every 500 m (shared across rows) ---
y_ticks = np.arange(0, depth_max+1, 500)
for ax in axes[:, 0]:
    ax.set_yticks(y_ticks)
    ax.tick_params(axis="y", labelsize=tick_size, pad=pad, length=8, width=3)

axes[1, 0].set_ylabel("Depth (m)", fontsize=label_size+20, labelpad=pad)
# ================================
# Loop over files and plot each row
# ================================
for row, fname in enumerate(files):
    ds = xr.open_dataset(fname)

    depth = ds["Depth"].values
    eps = ds["Eps_EG03"].values
    k = ds["Kv_EG03"].values
    n2 = ds["N2_ADCP"].values

    # --- turbulence scales ---
    eta = (nu**3 / eps) ** 0.25
    L_O = (eps / (n2**1.5)) ** 0.5
    u_eta = (nu * eps) ** 0.25
    u_O = (eps * L_O) ** (1.0 / 3.0)

    # conversions
    eta_cm = eta * 100.0
    L_O_cm = L_O * 100.0
    u_eta_day = u_eta * sec_per_day
    u_O_day = u_O * sec_per_day


    # ================================
    # Print statistical distributions
    # ================================
    # Mask data for first 1500 meters
    mask = depth <= 1500
    L_O_cm_masked = L_O_cm[mask]
    u_O_day_masked = u_O_day[mask]
    
    # ================================
    # Print statistical distributions
    # ================================
    '''
    print(f"\nStatistics for file: {fname.split('/')[-1]} ({row_titles[row]})")
    
    # L_O
    print("L_O (cm) [0-1500 m]:")
    print(f"  Mean: {np.nanmean(L_O_cm_masked):.2f}")
    print(f"  Median: {np.nanmedian(L_O_cm_masked):.2f}")
    print(f"  Std: {np.nanstd(L_O_cm_masked):.2f}")
    print(f"  Min: {np.nanmin(L_O_cm_masked):.2f}")
    print(f"  Max: {np.nanmax(L_O_cm_masked):.2f}")
    print(f"  25th percentile: {np.nanpercentile(L_O_cm_masked, 25):.2f}")
    print(f"  75th percentile: {np.nanpercentile(L_O_cm_masked, 75):.2f}")
    
    # u_O
    print("u_O (m/day) [0-1500 m]:")
    print(f"  Mean: {np.nanmean(u_O_day_masked):.2f}")
    print(f"  Median: {np.nanmedian(u_O_day_masked):.2f}")
    print(f"  Std: {np.nanstd(u_O_day_masked):.2f}")
    print(f"  Min: {np.nanmin(u_O_day_masked):.2f}")
    print(f"  Max: {np.nanmax(u_O_day_masked):.2f}")
    print(f"  25th percentile: {np.nanpercentile(u_O_day_masked, 25):.2f}")
    print(f"  75th percentile: {np.nanpercentile(u_O_day_masked, 75):.2f}")
    '''

    
    # (A) N2
    ax = axes[row, 0]
    ax.plot(n2, depth, color="purple", linewidth=plot_line_w, zorder=2)
    ax.set_xscale("log")
    if row == 2:
        ax.set_xlabel(r"$N^2$ (s$^{-2}$)", fontsize=label_size+20, labelpad = pad)
    else:
        ax.set_xlabel("")
    ax.set_ylim(0, depth_max)
    ax.invert_yaxis()
    ax.grid(True, which="both", ls="--", linewidth=grid_line, alpha=0.7, zorder=1)
    ax.tick_params(axis="y", labelsize=tick_size, pad=pad, length=8, width=3)
    ax.tick_params(axis="x", labelsize=tick_size, pad=pad, length=8, width=3)
    ax.text(0.03, 0.95, "a", transform=ax.transAxes,
            fontsize=label_size+ 40, fontfamily="serif", fontweight="bold")
    for spine in ax.spines.values():
        spine.set_linewidth(line_w)


    # (B) epsilon
    ax = axes[row, 1]
    ax.plot(eps, depth, color="green", linewidth=plot_line_w, zorder=2)
    ax.set_xscale("log")
    if row == 2:
        ax.set_xlabel(r"$\epsilon$ (m$^2$ s$^{-3}$)", fontsize=label_size+20, labelpad = pad)
    else:
        ax.set_xlabel("")        

    
    ax.set_ylim(0, depth_max)
    ax.invert_yaxis()
    ax.grid(True, which="both", ls="--", linewidth=grid_line, alpha=0.7, zorder=1)
    ax.tick_params(axis="y", labelsize=tick_size, pad=pad, length=8, width=3)
    ax.tick_params(axis="x", labelsize=tick_size, pad=pad, length=8, width=3)
    ax.text(0.03, 0.95, "b", transform=ax.transAxes,
            fontsize=label_size + 40, fontfamily="serif", fontweight="bold")
    for spine in ax.spines.values():
        spine.set_linewidth(line_w)

    # (C) eta and L_O
    ax = axes[row, 2]
    #ax.plot(eta_cm, depth, color="blue", linewidth=plot_line_w, label=r"$\eta$")
    ax.plot(L_O_cm, depth, color="red", linewidth=plot_line_w, label=r"$L_O$")
    ax.set_xscale("log")
    if row == 2:
        ax.set_xlabel("$L_{0}$ (cm)", fontsize=label_size+20, labelpad = pad)
    else:
        ax.set_xlabel("")
    ax.set_ylim(0, depth_max)
    ax.invert_yaxis()
    ax.grid(True, which="both", ls="--", linewidth=grid_line, alpha=0.7, zorder=1)
    ax.tick_params(axis="y", labelsize=tick_size, pad=pad, length=8, width=3)
    ax.tick_params(axis="x", labelsize=tick_size, pad=pad, length=8, width=3)
    ax.text(0.03, 0.95, "c", transform=ax.transAxes,
            fontsize=label_size+ 40, fontfamily="serif", fontweight="bold")
    for spine in ax.spines.values():
        spine.set_linewidth(line_w)
    #ax.legend(fontsize=label_size-10, loc="best")

    # (D) u_eta and u_O
    ax = axes[row, 3]
    #ax.plot(u_eta_day, depth, color="blue", linewidth=plot_line_w, label=r"$u_\eta$")
    ax.plot(u_O_day, depth, color="blue", linewidth=plot_line_w, label=r"$u_O$")
    ax.set_xscale("log")
    vel_ticks = [5, 10, 30, 50, 100]
    ax.set_xticks(vel_ticks)
    ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
    ax.set_xlim(min(vel_ticks), max(vel_ticks))

    if row == 2:
        ax.set_xlabel(r"$u_{0}$ (m day$^{-1}$)", fontsize=label_size+20, labelpad = pad)
    else: 
        ax.set_xlabel("")
    ax.set_ylim(0, depth_max)
    ax.invert_yaxis()
    ax.grid(True, which="both", ls="--", linewidth=grid_line, alpha=0.7, zorder=1)
    ax.tick_params(axis="y", labelsize=tick_size, pad=pad, length=8, width=3)
    ax.tick_params(axis="x", labelsize=tick_size, pad=pad, length=8, width=3)
    ax.text(0.03, 0.95, "d", transform=ax.transAxes,
            fontsize=label_size+ 40, fontfamily="serif", fontweight="bold")
    
    for spine in ax.spines.values():
        spine.set_linewidth(line_w)
    #ax.legend(fontsize=label_size-10, loc="best")


fig.text(0.1, 0.71,
    "PSS1",
    fontsize=label_size + 40,
    ha='center', va='bottom', color='black', 
    bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.4')
            )

fig.text(0.1, 0.385,
    "PSS2",
    fontsize=label_size + 40,
    ha='center', va='bottom', color='black', 
    bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.4')
            )

fig.text( 0.1, 0.06,
    "PSS4",
    fontsize=label_size + 40,
    ha='center', va='bottom', color='black', 
    bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.4')
            )

plt.tight_layout()
plt.subplots_adjust(hspace=0.1) 
plt.savefig('/home/aaccardo/these_alex/Papers/Rising_stars/Figure_S6/Figure_S6.png', dpi = 300)
plt.show()

In [5]:
# ================================
# Merge all profiles for 0-1500 m
# ================================
L_O_all = []
u_O_all = []

for row, fname in enumerate(files):
    ds = xr.open_dataset(fname)

    depth = ds["Depth"].values
    eps = ds["Eps_EG03"].values
    n2 = ds["N2_ADCP"].values

    # --- turbulence scales ---
    L_O = (eps / (n2**1.5)) ** 0.5
    u_O = (eps * L_O) ** (1.0 / 3.0)

    # conversions
    L_O_cm = L_O * 100.0
    u_O_day = u_O * sec_per_day

    # Mask for first 1500 m
    mask = depth <= 1500
    L_O_all.append(L_O_cm[mask])
    u_O_all.append(u_O_day[mask])

# Concatenate all arrays
L_O_all = np.concatenate(L_O_all)
u_O_all = np.concatenate(u_O_all)

# ================================
# Print merged statistics
# ================================
print("\n=== Merged statistics for all profiles (0-1500 m) ===")
print("L_O (cm):")
print(f"  Mean: {np.nanmean(L_O_all):.2f}")
print(f"  Median: {np.nanmedian(L_O_all):.2f}")
print(f"  Std: {np.nanstd(L_O_all):.2f}")
print(f"  Min: {np.nanmin(L_O_all):.2f}")
print(f"  Max: {np.nanmax(L_O_all):.2f}")
print(f"  25th percentile: {np.nanpercentile(L_O_all, 25):.2f}")
print(f"  75th percentile: {np.nanpercentile(L_O_all, 75):.2f}")

print("u_O (m/day):")
print(f"  Mean: {np.nanmean(u_O_all):.2f}")
print(f"  Median: {np.nanmedian(u_O_all):.2f}")
print(f"  Std: {np.nanstd(u_O_all):.2f}")
print(f"  Min: {np.nanmin(u_O_all):.2f}")
print(f"  Max: {np.nanmax(u_O_all):.2f}")
print(f"  25th percentile: {np.nanpercentile(u_O_all, 25):.2f}")
print(f"  75th percentile: {np.nanpercentile(u_O_all, 75):.2f}")


=== Merged statistics for all profiles (0-1500 m) ===
L_O (cm):
  Mean: 15.08
  Median: 12.86
  Std: 8.09
  Min: 6.12
  Max: 33.31
  25th percentile: 7.34
  75th percentile: 22.31
u_O (m/day):
  Mean: 27.73
  Median: 24.05
  Std: 10.83
  Min: 16.23
  Max: 53.52
  25th percentile: 17.86
  75th percentile: 33.93
